In [0]:
# Databricks notebook source
# src/landing/landing_simulator.py

import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Define parameter widgets for orchestration
dbutils.widgets.text("catalog", "meridian_t02", "Target Catalog")
dbutils.widgets.text("batch_id", "1", "Batch Number (1, 2, or 3)")
dbutils.widgets.text("volume_path", "/Volumes/meridian_t02/landing/raw_files", "Managed Volume Path")

catalog = dbutils.widgets.get("catalog")
batch_id = int(dbutils.widgets.get("batch_id"))
volume_base = dbutils.widgets.get("volume_path")

print(f"Executing Landing Simulator: Catalog={catalog}, Batch={batch_id}, Base Path={volume_base}")

# 2. Source tables reference
src_clickstream = spark.table("samples.wanderbricks.clickstream")
src_reviews     = spark.table("samples.wanderbricks.reviews")
src_users       = spark.table("samples.wanderbricks.users")
src_properties  = spark.table("samples.wanderbricks.properties")
src_bookings    = spark.table("samples.wanderbricks.bookings")

def write_batch_files(df, entity_name, batch_num):
    """Writes a DataFrame as JSON files into the assigned entity batch folder in the volume."""
    target_path = f"{volume_base}/{entity_name}/batch_{batch_num}"
    (df.write
       .format("json")
       .mode("overwrite")
       .save(target_path))
    print(f"Successfully landed {entity_name} (Batch {batch_num}) -> {target_path} [{df.count()} rows]")

# -------------------------------------------------------------------------
# ENTITY 1: PROPERTIES (Dimension)
# Landed in Batch 1 only
# -------------------------------------------------------------------------
if batch_id == 1:
    write_batch_files(src_properties, "properties", 1)

# -------------------------------------------------------------------------
# ENTITY 2: USERS (Dimension with SCD Type 2 Evolution)
# Batch 1: Baseline users
# Batch 2: Subset of users update user_type (e.g. guest -> host)
# Batch 3: Another subset updates user_type (e.g. host -> premium/vip)
# -------------------------------------------------------------------------
if batch_id == 1:
    write_batch_files(src_users, "users", 1)

elif batch_id == 2:
    # Mutate 15% of users to simulate user_type progression
    updated_users_b2 = (
        src_users.filter(F.col("user_id") % 10 in [1, 2])
                 .withColumn("user_type", F.lit("host_partner"))
                 .withColumn("updated_at", F.current_timestamp())
    )
    write_batch_files(updated_users_b2, "users", 2)

elif batch_id == 3:
    # Mutate another group to demonstrate multi-version SCD2 history
    updated_users_b3 = (
        src_users.filter(F.col("user_id") % 10 in [2, 3])
                 .withColumn("user_type", F.lit("vip_host"))
                 .withColumn("updated_at", F.current_timestamp())
    )
    write_batch_files(updated_users_b3, "users", 3)

# -------------------------------------------------------------------------
# FACT SLICING & ANOMALY INJECTION HELPER
# Facts are split into 3 disjoint chronological slices using ntile(3)
# -------------------------------------------------------------------------
def slice_fact_disjoint(df, time_col, target_slice):
    w = Window.orderBy(time_col)
    return df.withColumn("batch_slice", F.ntile(3).over(w)).filter(F.col("batch_slice") == target_slice).drop("batch_slice")

# Slice facts for current batch
batch_clicks = slice_fact_disjoint(src_clickstream, "timestamp", batch_id)
batch_reviews = slice_fact_disjoint(src_reviews, "created_at", batch_id)
batch_bookings = slice_fact_disjoint(src_bookings, "created_at", batch_id)

# -------------------------------------------------------------------------
# INJECT CONTROLLED ANOMALIES FOR SILVER QUALITY CONTRACT
# -------------------------------------------------------------------------
# Anomaly 1 (Null keys): Nullify user_id on a small fraction of clickstream
bad_clicks = batch_clicks.withColumn(
    "user_id", 
    F.when(F.rand(seed=42) < 0.02, F.lit(None).cast("bigint")).otherwise(F.col("user_id"))
)

# Anomaly 2 (Invalid range): Out-of-bounds ratings (rating = 99.0) in reviews
bad_reviews = batch_reviews.withColumn(
    "rating", 
    F.when(F.rand(seed=42) < 0.03, F.lit(99.0)).otherwise(F.col("rating"))
)

# Anomaly 3 (Duplicates & Negative financials): Injected duplicate and invalid amounts in bookings
sample_dupes = batch_bookings.limit(5)
bad_bookings = batch_bookings.unionByName(sample_dupes).withColumn(
    "total_amount",
    F.when(F.rand(seed=42) < 0.02, F.lit(-50.0).cast("float")).otherwise(F.col("total_amount"))
)

# Land the fact batches into Managed Volume
write_batch_files(bad_clicks, "clickstream", batch_id)
write_batch_files(bad_reviews, "reviews", batch_id)
write_batch_files(bad_bookings, "bookings", batch_id)

print(f"Batch {batch_id} landing complete.")